# Train the NCM-iWSP-GPT1 prefix level-0 head (new notebook)

This notebook trains **only** the Phase 1 prefix→level-0 head. The tokenizer and the v3 generator stay frozen. It does not train NCM-GPT1, and it does not edit or rerun the kernels `nicholashuihk/ncm-gpt-1-train` or `nicholashuihk/ncm-iwsp-gpt1-finetune`. Those kernels are attached read-only: packed windows from the former, `latest.pt` from the latter.

Repo copies of this notebook must keep `NICHOLAS_CONFIRMED = False`. Only a pushed Kaggle kernel that Nicholas has authorized may set it to `True`.

Version 1 failed at dry-plan with `No module named ncm_iwsp_gpt1` because `sys.path` in the notebook process is not inherited by `python -m` subprocesses. This notebook now puts the repo on `PYTHONPATH`, runs those commands from the repo root, and exits if a fresh process still cannot import the package.

This is continuation, not a chatbot. Source is ncm-iwsp-gpt1 `f35e352` plus a small time-budget flag so a short remaining GPU quota can still write `prefix_level0.pt`, `train_log.jsonl`, and `summary.json` (val CE at T=1/4/full vs unigram).


In [ ]:
NICHOLAS_CONFIRMED = False
print('NICHOLAS_CONFIRMED', NICHOLAS_CONFIRMED)


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

kaggle_input = Path("/kaggle/input")
kaggle_working = Path("/kaggle/working")
extracted = kaggle_working / "ncm-iwsp-gpt1"
REQUIRED = (
    "ncm_iwsp_gpt1/prefix_level0.py",
    "ncm_iwsp_gpt1/cli.py",
    "ncm_iwsp_gpt1/__main__.py",
    "ncm_iwsp_gpt1/chat.py",
    "ncm_iwsp_gpt1/finetune.py",
    "ncm_iwsp_gpt1/models/vqvae.py",
    "configs/target.yaml",
)


def list_input_tree():
    lines = []
    if not kaggle_input.is_dir():
        lines.append("/kaggle/input is missing")
        return lines
    top = sorted(path.name + ("/" if path.is_dir() else "") for path in kaggle_input.iterdir())
    lines.append("top-level /kaggle/input: " + (", ".join(top) if top else "(empty)"))
    return lines


def find_repo_root(*roots):
    for root in roots:
        if not root.is_dir():
            continue
        for path in root.rglob("prefix_level0.py"):
            if path.parent.name != "ncm_iwsp_gpt1":
                continue
            cand = path.parent.parent
            missing = [rel for rel in REQUIRED if not (cand / rel).is_file()]
            if not missing:
                return cand
    return None


def find_named(root, name, must_contain=None, must_not_contain=None):
    hits = [path for path in Path(root).rglob(name) if path.is_file()]
    if must_contain:
        hits = [path for path in hits if must_contain in str(path)]
    if must_not_contain:
        hits = [path for path in hits if must_not_contain not in str(path)]
    return hits


def put_repo_on_pythonpath(root):
    """Make this process and later `python -m` subprocesses import ncm_iwsp_gpt1."""
    root = str(Path(root).resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    parts = [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
    if root not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([root, *parts]) if parts else root
    print("repo_on_sys.path", root in sys.path, flush=True)
    print("PYTHONPATH", os.environ["PYTHONPATH"], flush=True)


def module_env():
    env = os.environ.copy()
    root = str(Path(ROOT).resolve())
    parts = [p for p in env.get("PYTHONPATH", "").split(os.pathsep) if p]
    if root not in parts:
        env["PYTHONPATH"] = os.pathsep.join([root, *parts]) if parts else root
    return env


print("\n".join(list_input_tree()), flush=True)
ROOT = find_repo_root(kaggle_input, kaggle_working)

if ROOT is None:
    zip_paths = []
    if kaggle_input.is_dir():
        zip_paths.extend(sorted(kaggle_input.rglob("repo.zip")))
    for zpath in zip_paths:
        extracted.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(extracted)
        print("extracted", zpath, "->", extracted, flush=True)
        ROOT = find_repo_root(extracted, kaggle_working, kaggle_input)
        if ROOT is not None:
            break

if ROOT is None:
    raise SystemExit(
        "Attach ncm-iwsp-gpt1 source that includes prefix_level0.py (main f35e352 or later). "
        "The older nicholashuihk/ncm-iwsp-gpt1-src dataset is stale.\n"
        + "\n".join(list_input_tree())
    )

commit_file = ROOT / "SOURCE_COMMIT.txt"
print("repo", ROOT, flush=True)
print("source_commit", commit_file.read_text(encoding="utf-8").splitlines()[0] if commit_file.is_file() else "missing SOURCE_COMMIT.txt", flush=True)
for rel in REQUIRED:
    print("required", rel, "ok", (ROOT / rel).stat().st_size, "bytes", flush=True)

put_repo_on_pythonpath(ROOT)
try:
    import ncm_iwsp_gpt1
    import ncm_iwsp_gpt1.prefix_level0  # noqa: F401
except Exception as exc:
    raise SystemExit(f"cannot import ncm_iwsp_gpt1 from {ROOT}: {type(exc).__name__}: {exc}") from exc
print("imported ncm_iwsp_gpt1 from", Path(ncm_iwsp_gpt1.__file__).resolve(), flush=True)

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import ncm_iwsp_gpt1, ncm_iwsp_gpt1.prefix_level0; print(ncm_iwsp_gpt1.__file__)",
    ],
    env=module_env(),
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print("subprocess_import", probe.stdout.strip(), flush=True)
if probe.returncode != 0:
    raise SystemExit(
        "a fresh python process still cannot import ncm_iwsp_gpt1 "
        f"(exit {probe.returncode}). stderr: {(probe.stderr or probe.stdout).strip() or 'empty'}"
    )

from ncm_iwsp_gpt1.finetune import find_packed_windows

tokenizer_hits = find_named(kaggle_input, "tokenizer.pt")
if not tokenizer_hits:
    raise SystemExit("no tokenizer.pt under /kaggle/input; attach nicholashuihk/ncm-iwsp-gpt1-v10-init")
# Prefer the v10 init dataset, not a random copy.
preferred = [p for p in tokenizer_hits if "ncm-iwsp-gpt1-v10-init" in str(p)]
tokenizer = (preferred or tokenizer_hits)[0]

latest_hits = find_named(kaggle_input, "latest.pt", must_contain="ncm-iwsp-gpt1-finetune")
if not latest_hits:
    latest_hits = find_named(kaggle_input, "latest.pt", must_not_contain="ncm-gpt-1-train")
if not latest_hits:
    raise SystemExit(
        "no finetune v3 latest.pt under /kaggle/input. "
        "Attach the complete kernel output of nicholashuihk/ncm-iwsp-gpt1-finetune."
    )
generator = latest_hits[0]
if generator.name != "latest.pt":
    raise SystemExit(f"expected latest.pt, found {generator}")

packed = find_packed_windows(kaggle_input)
out = Path("/kaggle/working/prefix-level0")
print("tokenizer", tokenizer, "bytes", tokenizer.stat().st_size, flush=True)
print("generator", generator, "bytes", generator.stat().st_size, flush=True)
print("packed", packed, "bytes", packed.stat().st_size, flush=True)
print("out", out, flush=True)
if tokenizer.stat().st_size < 100_000_000:
    raise SystemExit(f"tokenizer.pt looks too small: {tokenizer.stat().st_size} bytes")
if generator.stat().st_size < 100_000_000:
    raise SystemExit(f"latest.pt looks too small: {generator.stat().st_size} bytes")
if packed.stat().st_size < 100_000_000:
    raise SystemExit(f"packed_windows.pt looks too small: {packed.stat().st_size} bytes")


In [ ]:
import torch
print("cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(index, props.name, "mem_gib", round(props.total_memory / 1024**3, 1))
else:
    raise SystemExit("CUDA is required for this kernel")


Dry-plan first, then train only the prefix head. Writes only under `/kaggle/working/prefix-level0`.


In [ ]:
import subprocess
import sys
from pathlib import Path

plan_command = [
    sys.executable,
    "-m",
    "ncm_iwsp_gpt1",
    "train-prefix-level0",
    "--config", str(Path(ROOT) / "configs" / "target.yaml"),
    "--tokenizer", str(tokenizer),
    "--generator", str(generator),
    "--packed", str(packed),
    "--out", "/kaggle/working/prefix-level0",
    "--max-steps", "20000",
    "--device", "auto",
    "--max-hours", "2.6",
    "--reserve-minutes", "12",
    "--dry-plan",
]
print("PYTHONPATH", module_env().get("PYTHONPATH"), flush=True)
print("cwd", ROOT, flush=True)
print("dry-plan", " ".join(plan_command), flush=True)
rc = subprocess.call(plan_command, env=module_env(), cwd=str(ROOT))
if rc != 0:
    raise SystemExit(rc)


In [ ]:
import subprocess
import sys
from pathlib import Path

if not NICHOLAS_CONFIRMED:
    raise SystemExit(
        "Nicholas must set NICHOLAS_CONFIRMED = True after reviewing the paths. "
        "Repo copies of this notebook stay False. This cell does not submit a Kaggle job."
    )

command = [
    sys.executable,
    "-m",
    "ncm_iwsp_gpt1",
    "train-prefix-level0",
    "--config", str(Path(ROOT) / "configs" / "target.yaml"),
    "--tokenizer", str(tokenizer),
    "--generator", str(generator),
    "--packed", str(packed),
    "--out", "/kaggle/working/prefix-level0",
    "--max-steps", "20000",
    "--device", "auto",
    "--max-hours", "2.6",
    "--reserve-minutes", "12",
]
print("PYTHONPATH", module_env().get("PYTHONPATH"), flush=True)
print("cwd", ROOT, flush=True)
print("running", " ".join(command), flush=True)
rc = subprocess.call(command, env=module_env(), cwd=str(ROOT))
if rc != 0:
    raise SystemExit(rc)


In [ ]:
import json
import shutil
from pathlib import Path

out = Path("/kaggle/working/prefix-level0")
for name in ("summary.json", "train_log.jsonl", "prefix_level0.pt"):
    path = out / name
    print(name, "missing" if not path.is_file() else f"{path} {path.stat().st_size} bytes", flush=True)

summary_path = out / "summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2), flush=True)
    print("val_ce_T1", summary.get("val_ce_T1"), flush=True)
    print("val_ce_T4", summary.get("val_ce_T4"), flush=True)
    print("val_ce_Tfull", summary.get("val_ce_Tfull"), flush=True)
    print("val_unigram_nll", summary.get("val_unigram_nll"), flush=True)
    print("stopped_early", summary.get("stopped_early"), "steps", summary.get("steps"), "max_steps", summary.get("max_steps"), flush=True)
    # Copy the small artifacts to /kaggle/working root for easier download.
    for name in ("summary.json", "train_log.jsonl"):
        src = out / name
        if src.is_file():
            shutil.copy2(src, Path("/kaggle/working") / name)
else:
    raise SystemExit("summary.json was not written; training did not finish cleanly")
